# 第70课 · Whisper 架构解析 — Log-Mel 输入、Transformer Encoder-Decoder、多任务头

**目标**：读懂 Whisper 论文的架构，理解每个设计选择与 Aurora DSP 基础的对应关系。

🔗 **Aurora 连接**：Whisper 的 80-dim log-Mel 输入就是 L46-L47 实现的 `aurora.audio.mel.mel_spectrogram(n_mels=80)`，参数 `win_len=400, hop=160` 对应 25 ms 窗口 / 10 ms 帧移。

← **上一课**　[L69 · CTC 前向算法](L69_ctc_forward.ipynb)

> 上节课学习了 **CTC 前向算法**：所有路径概率求和，O(T×2L) 纯 NumPy 实现。  
> 本课将探讨 **Whisper 架构解析**。

## 学习目标

学完本节，你应该能够：

1. 理解 Whisper 80-dim log-Mel 输入的生成过程及与 Aurora DSP 的参数对齐（`n_fft=400, hop=160`）
2. 描述编码器 Conv1d×2 + Transformer 的形状变换链（输入 3000→Conv 降采样→1500→Transformer→(1500, d_model)）
3. 解释交叉注意力中 Q 来自解码器、K/V 来自编码器，以及注意力权重 shape `(B, T_dec, T_enc)`
4. 列出 Whisper 多任务头的四种输出及条件 token 前缀机制
5. 实现 `whisper_preprocess`，通过 30 s 静音、5 s 短音频补零、60 s 长音频截断三组断言

## Whisper / CTC 连接提示

Whisper 用的是交叉熵（cross-entropy）训练，而不是 CTC；L68–L69 则是理解“序列对齐基线”的前一站。
先把 CTC 的单调路径和 log 域 DP 看懂，再看 Whisper 就会知道它为什么选择 token-by-token 的自回归解码。

## 本课剧情：Whisper 是怎么做到"听懂99种语言"的？

OpenAI 2022 年发布的 Whisper，在没有任何特殊工程的情况下，能转写 99 种语言的语音。秘密是什么？

**答案：超大规模弱监督 + Encoder-Decoder + 多任务头。**

Whisper 把整个 ASR 流水线压缩成一个架构：

```
音频 (16kHz)
   ↓ log-Mel (n_fft=400, hop=160, n_mels=80, 30s→3000帧)
   ↓ 2×Conv1D 降采样 → 1500步向量序列
   ↓ Transformer Encoder (多层 self-attention)
   ↓ 上下文向量 (B, 1500, d_model)
   ↓ Transformer Decoder (cross-attention到编码器)
   ↓ BPE token 概率 (B, T_dec, vocab_size)
   ↓ 文字
```

**三个关键设计**：

| 设计 | 目的 |
|---|---|
| 固定30秒窗口 (3000帧) | 统一输入形状，无需变长处理 |
| 2×Conv1D降采样 (stride=2) | 3000帧→1500步，减少 decoder 注意力计算量 |
| 多任务 token (`<en>`,`<transcribe>`,`<translate>`) | 同一模型完成转写/翻译/VAD，无需多个模型 |

**Aurora 连接**：`aurora.audio.mel.mel_spectrogram()` 与 Whisper 官方参数完全对齐（n_mels=80, n_fft=400, hop=160, sr=16000）——本节 `whisper_preprocess()` 直接调用它。

本节任务：实现 `whisper_preprocess(wav)` — 音频→`(1, 80, 3000)` Tensor。

In [ ]:
import torch
import numpy as np
from aurora.audio.mel import mel_spectrogram, power_to_db


## 1. Whisper 输入：80-dim log-Mel，shape `(80, 3000)`

Whisper 固定使用 **30 秒**音频窗口（不足补零，超出截断）。采样率 16 kHz，所以 30 s = 480 000 个采样点。STFT 参数：`n_fft=400`（25 ms 窗口）、`hop=160`（10 ms 帧移）。帧数为 `480000 // 160 = 3000`，mel 维度固定 80。

```
帧数公式（Whisper 风格，先补零再切）：
  n_frames = n_samples / hop = 480000 / 160 = 3000
```

Whisper 的 log-Mel 归一化（normalization）：

```
log_mel = log10(max(mel_power, 1e-10))
log_mel = max(log_mel, log_mel.max() - 8.0)
log_mel = (log_mel + 4.0) / 4.0
```

这把动态范围压缩到 `[-1, 1]` 附近，让模型训练更稳定。Aurora 的 `power_to_db` 做的是 `10*log10`（dB），转换时只需除以 10 再做归一化。

In [ ]:
# 演示：验证帧数公式
sr = 16000
n_seconds = 30
hop = 160
n_samples = sr * n_seconds
n_frames = n_samples // hop
print(f"30 s @ 16 kHz = {n_samples} 采样点")
print(f"hop={hop} → 帧数 = {n_samples} / {hop} = {n_frames}")
print(f"Whisper 输入 shape: (80, {n_frames})")


## 2. 编码器：Conv1D × 2 + Transformer → 1500 步上下文向量

Whisper 编码器对 `(80, 3000)` 的 log-Mel 做两层 **Conv1D 降采样**，再送入 Transformer。

```
Conv1d(80 → d, kernel=3, padding=1, stride=1)   # 3000 → 3000
Conv1d(d  → d, kernel=3, padding=1, stride=2)   # 3000 → 1500
```

两层卷积总降采样为 **2 倍**：stride=1 保持 3000 帧，stride=2 压缩至 1500 帧：

```
层1: stride=1  → 时间维 3000
层2: stride=2  → 时间维 1500
```

注意：有些实现的第一层也加 stride=2，变成 3000 → 1500 → 750。Whisper 官方代码（openai/whisper）的输出是 **1500** 步，不是 750；"750" 是指 whisper-large 以外的小模型 ctx 窗口（max_source_positions）——编码器实际输出 `(1500, d_model)` 的上下文序列。

```
编码器输出 shape: (B, 1500, d_model)
  d_model: tiny=384, base=512, small=768, medium=1024, large=1280
```

每个 Transformer 层包含：**自注意力（self-attn）→ 前馈网络（FFN）**，编码器不用 causal mask（每帧可以看全局声学上下文）。

In [ ]:
# 演示：用 PyTorch 手写一个极简 Whisper 编码器片段（只展示 Conv 降采样）
import torch.nn as nn

d_model = 512  # Whisper-base
conv1 = nn.Conv1d(80, d_model, kernel_size=3, padding=1, stride=1)
conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, stride=2)

dummy_mel = torch.zeros(1, 80, 3000)   # (B, n_mels, T)
x1 = torch.gelu(conv1(dummy_mel))      # (1, 512, 3000)
x2 = torch.gelu(conv2(x1))             # (1, 512, 1500)
print(f"输入: {tuple(dummy_mel.shape)} → Conv1 → {tuple(x1.shape)} → Conv2 → {tuple(x2.shape)}")
print(f"编码器将继续接 {x2.shape[-1]} 步的 Transformer 层")


## 3. 解码器：自回归（autoregressive，AR） Transformer + 交叉注意力 → BPE token

解码器是标准的**自回归 causal Transformer**，与 GPT 结构类似，但多了一层 **交叉注意力（cross-attention）**，其 Key/Value 来自编码器输出。

```
每个解码器层的顺序：
  1. 自注意力（causal mask，只看已生成的 token）
  2. 交叉注意力（Q 来自解码器，K/V 来自编码器 1500 步）
  3. FFN
```

词表（vocabulary）大小：**51865**（多语言模型），使用 tiktoken BPE。特殊 token：`<|startoftranscript|>`、语言标签 `<|zh|>`、任务标签 `<|transcribe|>`、`<|notimestamps|>` 或时间戳 token（`<|0.00|>` 到 `<|30.00|>`）。

解码过程：

```
[SOT] → [语言] → [任务] → [内容 token_1] → [token_2] → ... → [EOT]
```

每步解码通过交叉注意力"看"整段 1500 步编码器输出，决定当前发什么音节。

In [ ]:
# 演示：交叉注意力的 shape 关系
B, T_enc, T_dec = 1, 1500, 20  # 编码器 1500 步，当前解码到第 20 个 token
d_model, n_heads = 512, 8
head_dim = d_model // n_heads

Q = torch.randn(B, T_dec, d_model)    # 来自解码器当前状态
K = torch.randn(B, T_enc, d_model)    # 来自编码器输出
V = torch.randn(B, T_enc, d_model)    # 来自编码器输出

# scaled dot-product attention: (B, T_dec, T_enc)
scale = head_dim ** 0.5
attn_weight = (Q @ K.transpose(-1,-2)) / scale
attn_weight = torch.softmax(attn_weight, dim=-1)
out = attn_weight @ V

print(f"Q: {tuple(Q.shape)}  K/V: {tuple(K.shape)}")
print(f"注意力权重: {tuple(attn_weight.shape)}  → 每个解码步看 1500 个编码步")
print(f"解码器输出: {tuple(out.shape)}")


## 4. 多任务头：一个解码器，四种输出

Whisper 的独特之处不仅是 encoder-decoder 架构，而是**同一个模型用特殊 token 条件化实现四个不同任务**：

| 任务 | 头输出 | 条件 token |
|------|--------|-----------|
| 语音识别（transcribe） | BPE 文字序列 | `[SOT][lang][transcribe][notimestamps]` |
| 语音翻译（translate） | 英文 BPE 序列 | `[SOT][lang][translate][notimestamps]` |
| 语言识别（language-ID） | lang token 概率 | `[SOT]` → 取下一 token 分布 |
| 带时间戳识别 | BPE + timestamp token 交错 | `[SOT][lang][transcribe]`（无 notimestamps） |

**关键设计**：所有任务共享同一个 **输出投影层**（`Linear(d_model, vocab_size)`），任务区分完全靠前缀 token 序列。这意味着训练时只需混合四类数据，推理时在 `[SOT]` 后拼接不同特殊 token 即可切换任务。

```
解码时 token 前缀（以 transcribe 为例）：
  [SOT=50258] [zh=50260] [transcribe=50359] [notimestamps=50362] → content tokens...
```

vocab_size = **51865**（50257 BPE + 1608 特殊 token），其中 timestamp token 占 1500 个（对应 0–30 s，0.02 s 精度）。

In [ ]:
# 演示：多任务头 token 条件化
# 特殊 token ID（来自 openai/whisper tokenizer）
SOT         = 50258   # start of transcript
LANG_ZH     = 50260   # 中文
TRANSCRIBE  = 50359   # 转录任务
TRANSLATE   = 50358   # 翻译任务
NO_TIMESTAMPS = 50362 # 不输出时间戳

# 三种任务的前缀序列
task_prefixes = {
    "识别（中文）": [SOT, LANG_ZH, TRANSCRIBE, NO_TIMESTAMPS],
    "翻译→英文":   [SOT, LANG_ZH, TRANSLATE, NO_TIMESTAMPS],
    "语言识别":    [SOT],            # 只看下一个 token 的分布
}
for task, prefix in task_prefixes.items():
    print(f"{task}: prefix tokens = {prefix}")

# 输出投影层 shape
import torch.nn as nn
vocab_size = 51865
d_model = 512  # Whisper-base
output_proj = nn.Linear(d_model, vocab_size, bias=False)
print(f"\n共享输出投影层: Linear({d_model}, {vocab_size}) = {sum(p.numel() for p in output_proj.parameters()):,} 参数")
print("所有任务共享同一 output_proj，切换任务只需改前缀 token。")

## 5. ✏️ 实现 `whisper_preprocess(wav, sr=16000)`

**签名**：
```python
def whisper_preprocess(wav: np.ndarray, sr: int = 16000) -> torch.Tensor:
    # 返回: (1, 80, 3000) float32 Tensor（batch_size=1，通道=1被省略）
```

**4步实现路线**：

| 步骤 | 操作 | 参数 |
|---|---|---|
| 1 | `mel_spectrogram(wav, sr, ...)` | n_fft=400, hop=160, n_mels=80 |
| 2 | `power_to_db(...)` | 转 dB（log-Mel），top_db=80 |
| 3 | pad/truncate → 固定3000帧 | `mel.T` shape=(T, 80)，T可能≠3000 |
| 4 | `torch.tensor(...).float().unsqueeze(0)` | (80, 3000) → (1, 80, 3000) |

**验收标准**：
- `whisper_preprocess(np.zeros(16000*30)).shape == (1, 80, 3000)`
- `dtype == torch.float32`
- 30秒以内音频 pad 到 3000 帧，超出截断

In [ ]:
def whisper_preprocess(wav: np.ndarray, sr: int = 16000) -> torch.Tensor:
    """
    将原始波形转换为 Whisper 所需的 log-Mel 特征张量。

    参数
    ----
    wav : np.ndarray, shape (T,)
        单声道 PCM 波形，采样率应为 sr。
    sr : int
        采样率，Whisper 默认 16000 Hz。

    返回
    ----
    torch.Tensor, shape (1, 80, 3000), dtype=float32
    """
    # ✏️ TODO: 步骤1 — 调用 aurora mel_spectrogram，参数 n_fft=400, hop_length=160, n_mels=80
    #          注意返回 shape 是 (n_frames, 80)，需要 .T 转置

    # ✏️ TODO: 步骤2 — log 归一化（Whisper 风格）
    #          log_mel = log10(max(mel, 1e-10))
    #          log_mel = max(log_mel, log_mel.max() - 8.0)
    #          log_mel = (log_mel + 4.0) / 4.0

    # ✏️ TODO: 步骤3 — pad 或 truncate 到 3000 帧（axis=1）

    # ✏️ TODO: 步骤4 — 转 torch.float32 tensor，unsqueeze(0)
    raise NotImplementedError("whisper_preprocess: 按步骤1-4实现本函数")


In [ ]:
# 检查：30 秒静音
feat = whisper_preprocess(np.zeros(16000 * 30))
assert feat.shape == (1, 80, 3000), f"shape 错误: {feat.shape}"
assert feat.dtype == torch.float32, f"dtype 错误: {feat.dtype}"
print(f"✅ shape={tuple(feat.shape)}, dtype={feat.dtype}")
print(f"✅ 值域: [{feat.min():.3f}, {feat.max():.3f}]（静音应趋向 -1）")

# 检查：短音频自动补零
feat_short = whisper_preprocess(np.zeros(16000 * 5))  # 5 秒
assert feat_short.shape == (1, 80, 3000), f"短音频补零失败: {feat_short.shape}"
print(f"✅ 5 秒短音频自动补零 → shape={tuple(feat_short.shape)}")

# 检查：长音频自动截断
feat_long = whisper_preprocess(np.zeros(16000 * 60))  # 60 秒
assert feat_long.shape == (1, 80, 3000), f"长音频截断失败: {feat_long.shape}"
print(f"✅ 60 秒长音频截断 → shape={tuple(feat_long.shape)}")


## 6. 参数实验：Aurora vs. Whisper 官方 log-Mel 的差异

对比 `aurora.audio.mel.mel_spectrogram` 与 openai/whisper 官方的 `log_mel_spectrogram` （`whisper/audio.py`），找出以下参数的差异：

| 参数 | Aurora 默认 | Whisper 官方 |
|------|-------------|-------------|
| `n_fft` | 1024 | **400** |
| `hop_length` | `n_fft // 4 = 256` | **160** |
| `n_mels` | 80 | 80（相同）|
| `fmin` | 0.0 | **0.0**（相同）|
| `fmax` | `sr / 2` | **sr / 2**（相同）|
| log 归一化 | `power_to_db`（dB，`10*log10`）| **`log10` + clip(-8) + /4 归一化** |
| 输出 shape | `(n_frames, n_mels)` | **`(n_mels, n_frames)`** |

**预期现象**：
- 用 Aurora 默认参数（`n_fft=1024, hop=256`）时，30 s 音频输出 `(1876, 80)` → 转置后 `(80, 1876)`，  不足 3000 帧，补零后与官方结果的**时间分辨率**不同（频率分辨率也更高）。
- 用 Whisper 参数（`n_fft=400, hop=160`）时，30 s 输出约 `(3000, 80)`，与官方对齐。
- 归一化方式不同：`power_to_db` 输出 dB 值（如 -80 ~ 0 dB），  而 Whisper 归一化后值域约 `[-1, 1]`。


In [ ]:
# 实验1：Aurora 默认参数 vs Whisper 参数的帧数差异
wav_30s = np.zeros(16000 * 30)

# Aurora 默认 (n_fft=1024, hop=256)
mel_aurora_default = mel_spectrogram(wav_30s, 16000, n_fft=1024, hop_length=256, n_mels=80)
print(f"Aurora 默认参数 → mel shape: {mel_aurora_default.shape}  (帧数={mel_aurora_default.shape[0]})")

# Whisper 参数 (n_fft=400, hop=160)
mel_whisper = mel_spectrogram(wav_30s, 16000, n_fft=400, hop_length=160, n_mels=80)
print(f"Whisper 参数   → mel shape: {mel_whisper.shape}  (帧数={mel_whisper.shape[0]})")

# 实验2：log 归一化对比
mel_raw = mel_spectrogram(wav_30s[:16000], 16000, n_fft=400, hop_length=160, n_mels=80)  # 1 秒

# Aurora power_to_db 风格
db = power_to_db(mel_raw)
print(f"\npower_to_db 值域: [{db.min():.1f}, {db.max():.1f}] dB")

# Whisper 风格
log_mel = np.log10(np.maximum(mel_raw, 1e-10))
log_mel = np.maximum(log_mel, log_mel.max() - 8.0)
log_mel = (log_mel + 4.0) / 4.0
print(f"Whisper 归一化 值域: [{log_mel.min():.3f}, {log_mel.max():.3f}]")
print("\n✅ 结论：喂给官方 Whisper 编码器必须用 n_fft=400, hop=160，并做 Whisper 风格归一化")


## 本课收束

本节实现了 `whisper_preprocess(wav)`，输出 `torch.Tensor` shape `(1, 80, 3000)`，是连接 Aurora DSP 模块与 Whisper 编码器的关键桥梁。核心依赖 `aurora.audio.mel.mel_spectrogram`，通过调整 `n_fft=400, hop_length=160` 对齐 Whisper 的声学参数，并加入 Whisper 风格 log 归一化（`log10 → clip → /4`）。下一节（L71）将实现 Whisper 的两种解码策略：贪婪解码（每步取最高概率 token）和宽度为 2 的 beam search，对比两者在玩具语言模型上的序列质量差异。

## ✏️ 闭卷推导检查格 — Whisper 数据流图

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：Whisper-small 参数：
- 输入：30秒音频，80-mel，帧移 10ms → 3000 帧
- Encoder：12 层 Transformer，d_model=768，8 头
- Decoder：输入 BOS token，自回归生成

画出数据流图并标注每一步的**张量形状**：

| 步骤 | 操作 | 输出形状 |
|------|------|---------|
| 1 | log-mel 输入 | (batch, 80, 3000) |
| 2 | Conv1 (kernel=3, stride=1, out=768) | (batch, ___, ___) |
| 3 | Conv2 (kernel=3, stride=2, out=768) | (batch, ___, ___) |
| 4 | Encoder 输出 | (batch, ___, ___) |
| 5 | Decoder 交叉注意力 K/V 来自 | 步骤 ___ 的输出 |

（在此处填表...）

In [ ]:
# 验证：Whisper 卷积层的输出形状计算（不需要加载模型权重）
import numpy as np

def conv1d_out(L_in, kernel, stride=1, padding=1):
    return (L_in + 2*padding - kernel) // stride + 1

T = 3000   # 3000 mel 帧
after_conv1 = conv1d_out(T, kernel=3, stride=1, padding=1)  # 保持长度
after_conv2 = conv1d_out(after_conv1, kernel=3, stride=2, padding=1)  # 下采样 2x

# Whisper-small encoder 序列长度 = 1500
assert after_conv1 == 3000, f"Conv1 输出 {after_conv1}，期望 3000"
assert after_conv2 == 1500, f"Conv2 输出 {after_conv2}，期望 1500"
print(f"Conv1 输出序列长度: {after_conv1}")
print(f"Conv2 输出序列长度: {after_conv2}")
print(f"✅ Whisper 维度推导正确：(batch, 768, 1500) 进入 Transformer Encoder")

In [ ]:
# ✏️ 本课自评
l70_review = {
    "whisper_3stage_understood":  None,  # 记住Whisper三阶段：log-Mel→Encoder→Decoder→BPE？True/False
    "frame_count_3000":           None,  # 理解30s@16kHz用hop=160→3000帧的计算？True/False
    "cross_attention_understood": None,  # 理解解码器cross-attention：Q来自decoder，K/V来自encoder？True/False
    "whisper_preprocess_impl":    None,  # whisper_preprocess()实现正确，shape/dtype验证通过？True/False
    "whiteboard_passed":          None,  # 白板挑战Whisper数据流图推导通关？True/False
}

unfilled = [k for k, v in l70_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l70_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L70 全部通关！进入 L71：Whisper 解码策略')

---

→ **下一课**　[L71 · Whisper 解码策略](L71_whisper_decoding.ipynb)

> 下节课将学习 **Whisper 解码策略**：贪婪解码与 beam search 从原理到实现。